In [3]:
import gzip
import shutil
from urllib.request import urlretrieve

# Step 1: Download
url = 'ftp://ita.ee.lbl.gov/traces/calgary_access_log.gz'
urlretrieve(url, 'calgary_access_log.gz')

# Step 2: Unzip
with gzip.open('calgary_access_log.gz', 'rb') as f_in:
    with open('calgary_access_log', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Dataset downloaded and extracted!")


Dataset downloaded and extracted!


In [4]:
import pandas as pd
import re
from datetime import datetime
from tqdm import tqdm


In [5]:
with open('calgary_access_log', 'r', encoding='latin1') as f:
    lines = f.readlines()
print(f"Total lines: {len(lines)}")

log_pattern = re.compile(
    r'(?P<host>\S+) \S+ \S+ \[(?P<timestamp>.*?)\] "(?P<request>.*?)" (?P<status>\d{3}) (?P<bytes>\S+)'
)


Total lines: 726739


In [6]:
data = []

for line in tqdm(lines):
    match = log_pattern.match(line)
    if match:
        entry = match.groupdict()
        
        # Parse request into method and filename
        try:
            parts = entry['request'].split()
            entry['method'] = parts[0] if len(parts) > 0 else None
            entry['filename'] = parts[1] if len(parts) > 1 else None
        except:
            entry['method'] = entry['filename'] = None

        # Parse timestamp to datetime
        entry['datetime'] = datetime.strptime(entry['timestamp'], "%d/%b/%Y:%H:%M:%S %z")
        entry['date'] = entry['datetime'].strftime('%d-%b-%Y')
        entry['hour'] = entry['datetime'].hour
        
        # Clean bytes
        entry['bytes'] = int(entry['bytes']) if entry['bytes'].isdigit() else 0
        
        # File extension
        entry['ext'] = re.findall(r'\.([a-zA-Z0-9]+)$', entry['filename'] or '')
        entry['ext'] = entry['ext'][0] if entry['ext'] else None

        data.append(entry)


100%|███████████████████████████████████████████████████████████████████████| 726739/726739 [00:43<00:00, 16770.35it/s]


In [7]:
df = pd.DataFrame(data)
df.head()


,host,timestamp,request,status,bytes,method,filename,datetime,date,hour,ext
0,local,24/Oct/1994:13:41:41 -0600,GET index.html HTTP/1.0,200,150,GET,index.html,1994-10-24 13:41:41-06:00,24-Oct-1994,13,html
1,local,24/Oct/1994:13:41:41 -0600,GET 1.gif HTTP/1.0,200,1210,GET,1.gif,1994-10-24 13:41:41-06:00,24-Oct-1994,13,gif
2,local,24/Oct/1994:13:43:13 -0600,GET index.html HTTP/1.0,200,3185,GET,index.html,1994-10-24 13:43:13-06:00,24-Oct-1994,13,html
3,local,24/Oct/1994:13:43:14 -0600,GET 2.gif HTTP/1.0,200,2555,GET,2.gif,1994-10-24 13:43:14-06:00,24-Oct-1994,13,gif
4,local,24/Oct/1994:13:43:15 -0600,GET 3.gif HTTP/1.0,200,36403,GET,3.gif,1994-10-24 13:43:15-06:00,24-Oct-1994,13,gif


In [10]:
# Q1: Count of total log records

total_requests = len(df)
print("Total log records:", total_requests)



Total log records: 724910


In [11]:
# Q2: Count of unique hosts

unique_hosts = df['host'].nunique()
print("Unique hosts:", unique_hosts)


Unique hosts: 2


In [12]:
# Q3: Date-wise unique filename counts

unique_filenames_per_date = df.groupby('date')['filename'].nunique().to_dict()
unique_filenames_per_date


{'01-Apr-1995': 438,
 '01-Aug-1995': 684,
 '01-Dec-1994': 271,
 '01-Feb-1995': 625,
 '01-Jan-1995': 88,
 '01-Jul-1995': 388,
 '01-Jun-1995': 591,
 '01-Mar-1995': 582,
 '01-May-1995': 467,
 '01-Nov-1994': 415,
 '01-Oct-1995': 554,
 '01-Sep-1995': 328,
 '02-Apr-1995': 466,
 '02-Aug-1995': 857,
 '02-Dec-1994': 325,
 '02-Feb-1995': 524,
 '02-Jan-1995': 141,
 '02-Jul-1995': 400,
 '02-Jun-1995': 515,
 '02-Mar-1995': 601,
 '02-May-1995': 702,
 '02-Nov-1994': 430,
 '02-Oct-1995': 873,
 '02-Sep-1995': 352,
 '03-Apr-1995': 795,
 '03-Aug-1995': 584,
 '03-Dec-1994': 189,
 '03-Feb-1995': 570,
 '03-Jan-1995': 311,
 '03-Jul-1995': 438,
 '03-Jun-1995': 401,
 '03-Mar-1995': 510,
 '03-May-1995': 609,
 '03-Nov-1994': 460,
 '03-Oct-1995': 848,
 '03-Sep-1995': 214,
 '04-Apr-1995': 821,
 '04-Aug-1995': 717,
 '04-Dec-1994': 213,
 '04-Feb-1995': 561,
 '04-Jan-1995': 333,
 '04-Jul-1995': 612,
 '04-Jun-1995': 371,
 '04-Mar-1995': 413,
 '04-May-1995': 684,
 '04-Nov-1994': 404,
 '04-Oct-1995': 893,
 '04-Sep-1995'

In [21]:
# Q4: Number of 404 response codes

df[df['status'] == '404'].shape[0]

print("Total 404 errors:", total_404s)


Total 404 errors: 23586


In [22]:
# Q5: Top 15 filenames with 404 responses

top_15_404_files = df[df['status'] == '404']['filename'].value_counts().head(15)
list(top_15_404_files.items())




[('index.html', 4737),
 ('4115.html', 902),
 ('1611.html', 649),
 ('5698.xbm', 585),
 ('710.txt', 408),
 ('2002.html', 259),
 ('2177.gif', 193),
 ('10695.ps', 161),
 ('6555.html', 153),
 ('487.gif', 152),
 ('151.html', 149),
 ('488.gif', 148),
 ('3414.gif', 148),
 ('40.html', 148),
 ('9678.gif', 142)]

In [23]:
# Q6: Top 15 file extensions with 404 responses

top_15_404_ext = df[df['status'] == '404']['ext'].value_counts().head(15)
list(top_15_404_ext.items())



[('html', 12213),
 ('gif', 7213),
 ('xbm', 826),
 ('ps', 754),
 ('jpg', 527),
 ('txt', 497),
 ('GIF', 135),
 ('htm', 109),
 ('cgi', 77),
 ('com', 46),
 ('Z', 41),
 ('dvi', 40),
 ('ca', 36),
 ('hmtl', 30),
 ('util', 29)]

In [ ]:
# Q7: Total bandwidth transferred per day for July 1995

# Ensure datetime column is properly converted
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

df_july = df[df['datetime'].dt.strftime('%b-%Y') == 'Jul-1995']
july_bandwidth = df_july.groupby('date')['bytes'].sum().to_dict()
july_bandwidth


In [ ]:
# Q8: Hourly request distribution

hourly_distribution = df['hour'].value_counts().sort_index().to_dict()
hourly_distribution


In [24]:
# Q9: Top 10 most requested filenames

top_10_files = df['filename'].value_counts().head(10)
list(top_10_files.items())



[('index.html', 140076),
 ('3.gif', 24006),
 ('2.gif', 23606),
 ('4.gif', 8018),
 ('244.gif', 5149),
 ('5.html', 5010),
 ('4097.gif', 4874),
 ('8870.jpg', 4493),
 ('6733.gif', 4278),
 ('8472.gif', 3843)]

In [ ]:
# Q10: HTTP response code distribution

response_distribution = df['status'].value_counts().to_dict()
response_distribution

